In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 파일들이 있는 폴더로 이동 (경로는 본인 설정에 맞게 수정)
%cd /content/

# 3. 코랩 로컬 영역(/content)에 압축 풀기 (속도 향상을 위해 로컬 사용)
!unzip -q /content/drive/MyDrive/안전모/공사현장_안전모.zip -d /content/drive/MyDrive/안전모/dataset/train/images
!unzip -q /content/drive/MyDrive/안전모/공사현장_안전모2.zip -d /content/drive/MyDrive/안전모/dataset/train/images
!unzip -q /content/drive/MyDrive/안전모/공사현장_안전모3.zip -d /content/drive/MyDrive/안전모/dataset/train/images
!unzip -q /content/drive/MyDrive/안전모/공사현장_안전모4.zip -d /content/drive/MyDrive/안전모/dataset/train/images

In [ ]:
import os
import json
import zipfile

# 1. 압축 해제 (파일명은 본인의 파일명에 맞게 수정하세요)
zip_path = '/content/drive/MyDrive/안전모/const_site_00001.zip'  # 업로드한 압축파일 경로
extract_path = '/content/drive/MyDrive/안전모/train 라벨 폴리곤' # 압축 풀 곳

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 2. 결과 저장 경로 설정
output_label_dir = '/content/drive/MyDrive/안전모/dataset/train/labels'
os.makedirs(output_label_dir, exist_ok=True)

# 클래스 맵핑 (안전모: 1, 머리: 0 등 프로젝트 설정에 맞춰주세요)
# 이제 실제 JSON 데이터의 클래스명과 일치시킵니다.
class_mapping = {"안전모": 1, "머리": 0} # '안전모'와 '머리'로 클래스 매핑 업데이트


In [ ]:
import random

def convert_to_yolo(json_data, output_path):
    # Ensure img_w and img_h are correctly extracted and are not zero
    if not json_data.get('images') or not json_data['images'][0].get('width') or not json_data['images'][0].get('height'):
        print(f"Warning: Image dimensions missing in JSON data for {output_path}. Skipping file.")
        return

    img_w = float(json_data['images'][0]['width'])
    img_h = float(json_data['images'][0]['height'])

    if img_w <= 0 or img_h <= 0:
        print(f"Warning: Invalid image dimensions (w={img_w}, h={img_h}) for {output_path}. Skipping file.")
        return

    yolo_lines = []
    annotations_processed = 0
    annotations_skipped_invalid_class = 0
    annotations_skipped_invalid_polygon = 0
    annotations_skipped_invalid_bbox = 0

    for ann in json_data['annotations']:
        cls_name = ann.get('class') # Use .get to safely access
        if cls_name not in class_mapping:
            annotations_skipped_invalid_class += 1
            # print(f"Debug: Skipped annotation in {os.path.basename(output_path)} due to unknown class: '{cls_name}'")
            continue

        cls_id = class_mapping[cls_name]

        polygon = ann.get('polygon')
        if not polygon or len(polygon) < 4: # A polygon needs at least 2 points (4 coordinates) to form a box
            annotations_skipped_invalid_polygon += 1
            print(f"Debug: Skipped annotation in {os.path.basename(output_path)} due to invalid polygon data: {polygon}") # Debug print for invalid polygon
            continue

        x_coords = [float(p) for p in polygon[0::2]] # Ensure coordinates are floats
        y_coords = [float(p) for p in polygon[1::2]]

        # Calculate Bounding Box
        # Clip to 0 and img_w-1/img_h-1 to ensure coordinates are within image boundaries
        xmin_raw = min(x_coords)
        ymin_raw = min(y_coords)
        xmax_raw = max(x_coords)
        ymax_raw = max(y_coords)

        xmin = max(0.0, xmin_raw)
        ymin = max(0.0, ymin_raw)
        xmax = min(img_w - 1, xmax_raw)
        ymax = min(img_h - 1, ymax_raw)

        # Check for valid bounding box after clipping
        if xmax <= xmin or ymax <= ymin:
            annotations_skipped_invalid_bbox += 1
            # print(f"Debug: Skipped annotation in {output_path} due to degenerate bbox after clipping: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}, img_w={img_w}, img_h={img_h}")
            continue

        # YOLO 포맷 변환 (정규화 및 중심점 계산)
        x_center = ((xmin + xmax) / 2.0) / img_w
        y_center = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        # Ensure normalized values are strictly within [0, 1] range
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        w = max(0.0, min(1.0, w))
        h = max(0.0, min(1.0, h))

        yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        annotations_processed += 1

    # Only write the file if there are valid annotations, otherwise create an empty file
    with open(output_path, 'w') as f:
        if yolo_lines:
            f.write("\n".join(yolo_lines))

    if annotations_skipped_invalid_class > 0 or annotations_skipped_invalid_polygon > 0 or annotations_skipped_invalid_bbox > 0:
        print(f"Summary for {os.path.basename(output_path)}: Processed {annotations_processed} annotations. Skipped: {annotations_skipped_invalid_class} (unknown class), {annotations_skipped_invalid_polygon} (invalid polygon), {annotations_skipped_invalid_bbox} (invalid bbox after clipping).")

# 3. 폴더 내 모든 JSON 파일 변환 실행
json_files = [f for f in os.listdir(extract_path) if f.endswith('.json')]

for j_file in json_files:
    with open(os.path.join(extract_path, j_file), 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 파일명 추출 (const_site_00001.json -> const_site_00001.txt)
    txt_filename = os.path.splitext(j_file)[0] + '.txt'
    save_path = os.path.join(output_label_dir, txt_filename)

    convert_to_yolo(data, save_path)

print(f"총 {len(json_files)}개의 라벨 변환 시도 완료!")

# --- Post-conversion checks (new cells below will handle these) ---

In [ ]:
# 4. 라벨 파일 생성 확인 및 샘플 출력

# 생성된 라벨 파일 목록
label_files = [f for f in os.listdir(output_label_dir) if f.endswith('.txt')]

print(f"총 생성된 라벨 파일 수: {len(label_files)} 개")

# 비어있지 않은 라벨 파일 수 확인
non_empty_label_files = 0
for l_file in label_files:
    with open(os.path.join(output_label_dir, l_file), 'r', encoding='utf-8') as f:
        content = f.read()
        if content.strip(): # 내용이 비어있지 않은 경우
            non_empty_label_files += 1

print(f"성공적으로 라벨이 변환된 (비어있지 않은) 파일 수: {non_empty_label_files} 개")

if non_empty_label_files == 0:
    print("경고: 생성된 라벨 파일 중 유효한 내용이 있는 파일이 없습니다. 라벨링 데이터 또는 변환 로직을 다시 확인해주세요.")

# 랜덤으로 5개 파일 내용 출력
print("\n--- 랜덤 샘플 라벨 파일 내용 (5개) ---")
if label_files:
    sample_files = random.sample(label_files, min(5, len(label_files)))
    for s_file in sample_files:
        print(f"\n파일: {s_file}")
        with open(os.path.join(output_label_dir, s_file), 'r', encoding='utf-8') as f:
            print(f.read().strip())
else:
    print("생성된 라벨 파일이 없습니다.")

In [ ]:
import yaml

# 데이터셋 경로 및 클래스 설정
data = {
    'train': '/content/drive/MyDrive/안전모/dataset/train/images',
    'val': '/content/drive/MyDrive/안전모/dataset/valid/images',
    'test': '/content/drive/MyDrive/안전모/dataset/test/images',
    'nc': 2, # 클래스 수 (Head, Helmet)
    'names': ['Head', 'Helmet'] # 0: 머리, 1: 안전모
}

# data.yaml 파일로 저장
with open('/content/drive/MyDrive/안전모/dataset/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("data.yaml 파일 생성이 완료되었습니다.")

### 검증(Validation) 라벨 파일 복사 (훈련 데이터셋에서)

In [ ]:
import os
import shutil

train_label_dir = '/content/drive/MyDrive/안전모/dataset/train/labels'
valid_image_dir = '/content/drive/MyDrive/안전모/dataset/valid/images'
valid_label_dir = '/content/drive/MyDrive/안전모/dataset/valid/labels'

os.makedirs(valid_label_dir, exist_ok=True)

# 검증 이미지 파일 목록 가져오기
valid_image_files = [f for f in os.listdir(valid_image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

files_copied_count = 0
for img_file in valid_image_files:
    # 이미지 파일명에서 라벨 파일명 유추 (확장자 변경)
    label_filename = os.path.splitext(img_file)[0] + '.txt'

    source_label_path = os.path.join(train_label_dir, label_filename)
    destination_label_path = os.path.join(valid_label_dir, label_filename)

    # 원본 라벨 파일이 존재하는지 확인 후 복사
    if os.path.exists(source_label_path):
        shutil.copy(source_label_path, destination_label_path)
        files_copied_count += 1
    else:
        print(f"경고: 훈련 라벨 폴더에 해당하는 라벨 파일이 없습니다: {label_filename}")

print(f"총 {files_copied_count}개의 라벨 파일이 '{valid_label_dir}'로 복사되었습니다.")

# 복사 후 다시 검증 라벨 파일 수 확인
validation_label_files_after_copy = [f for f in os.listdir(valid_label_dir) if f.endswith('.txt')]
non_empty_validation_label_files_after_copy = 0
for l_file in validation_label_files_after_copy:
    with open(os.path.join(valid_label_dir, l_file), 'r', encoding='utf-8') as f:
        content = f.read()
        if content.strip():
            non_empty_validation_label_files_after_copy += 1

print(f"\n복사 후 총 생성된 검증 라벨 파일 수: {len(validation_label_files_after_copy)} 개")
print(f"복사 후 성공적으로 라벨이 변환된 (비어있지 않은) 검증 파일 수: {non_empty_validation_label_files_after_copy} 개")

# 랜덤으로 5개 파일 내용 출력
print("\n--- 복사 후 랜덤 샘플 검증 라벨 파일 내용 (5개) ---")
if validation_label_files_after_copy:
    sample_files_after_copy = random.sample(validation_label_files_after_copy, min(5, len(validation_label_files_after_copy)))
    for s_file in sample_files_after_copy:
        print(f"\n파일: {s_file}")
        with open(os.path.join(valid_label_dir, s_file), 'r', encoding='utf-8') as f:
            print(f.read().strip())
else:
    print("복사된 검증 라벨 파일이 없습니다.")

In [ ]:
# TPU 관련 라이브러리 설치 및 확인
!pip install ultralytics

In [ ]:
drive_model_path = '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring/weights/best.pt'
print(f"모델 가중치 경로가 설정되었습니다: {drive_model_path}")

In [ ]:
from ultralytics import YOLO

# 2. 모델 로드 (Pre-trained 모델 또는 기존 학습 모델 사용)
# drive_model_path 변수는 이미 이전 셀에서 정의되었습니다.
# drive_model_path = '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring/weights/best.pt'
model = YOLO(drive_model_path) # 기존에 학습된 모델을 로드하여 추가 학습(fine-tuning)을 진행합니다.

# 3. 학습 시작
model.train(
    data='/content/drive/MyDrive/안전모/dataset/data.yaml', # 위에서 만든 설정 파일 경로
    epochs=80,                        # 학습 반복 횟수
    imgsz=640,                         # 입력 이미지 크기
    batch=16,                          # 배치 사이즈 (메모리 부족 시 8로 줄이세요)
    cache=True,
    workers=8,
    amp=True,
    device=0,                          # GPU 사용 (0번)
    project='Helmet_Detection',        # 결과 저장 폴더명
    name='yolo26_safety_monitoring_finetune', # 세부 결과 폴더명 (fine-tuning을 위한 새로운 이름)
    exist_ok=True
)

### 검증(Validation) 데이터셋 라벨 파일 확인

In [ ]:
import os
import random

# 검증 데이터셋 라벨 경로 설정
validation_label_dir = '/content/drive/MyDrive/안전모/dataset/valid/labels'

# 디렉토리가 존재하는지 확인하고 없으면 생성
os.makedirs(validation_label_dir, exist_ok=True)

# 생성된 라벨 파일 목록
validation_label_files = [f for f in os.listdir(validation_label_dir) if f.endswith('.txt')]

print(f"총 생성된 검증 라벨 파일 수: {len(validation_label_files)} 개")

# 비어있지 않은 라벨 파일 수 확인
non_empty_validation_label_files = 0
for l_file in validation_label_files:
    with open(os.path.join(validation_label_dir, l_file), 'r', encoding='utf-8') as f:
        content = f.read()
        if content.strip(): # 내용이 비어있지 않은 경우
            non_empty_validation_label_files += 1

print(f"성공적으로 라벨이 변환된 (비어있지 않은) 검증 파일 수: {non_empty_validation_label_files} 개")

if non_empty_validation_label_files == 0:
    print("경고: 생성된 검증 라벨 파일 중 유효한 내용이 있는 파일이 없습니다. 라벨링 데이터 또는 변환 로직을 다시 확인해주세요.")

# 랜덤으로 5개 파일 내용 출력
print("\n--- 랜덤 샘플 검증 라벨 파일 내용 (5개) ---")
if validation_label_files:
    sample_files = random.sample(validation_label_files, min(5, len(validation_label_files)))
    for s_file in sample_files:
        print(f"\n파일: {s_file}")
        with open(os.path.join(validation_label_dir, s_file), 'r', encoding='utf-8') as f:
            print(f.read().strip())
else:
    print("생성된 검증 라벨 파일이 없습니다.")


### 중단된 학습 이어하기 (Resume Training)

학습 도중 Colab 런타임이 끊기거나, 사용자가 수동으로 중단했을 경우, `resume=True` 옵션을 사용하여 마지막 체크포인트부터 학습을 이어갈 수 있습니다. `ultralytics`는 자동으로 해당 `project/name` 폴더 내에서 가장 최신 체크포인트(`last.pt`)를 찾아 학습을 재개합니다.

In [ ]:
from ultralytics import YOLO
import os

# 이전 미세 조정 학습 세션의 last.pt가 임시 Colab 디렉토리(/content)에 있었고
# 런타임 재시작으로 인해 손실되었으므로,
# Google Drive에 저장된 *초기* 학습의 최적 모델에서 미세 조정 프로세스를 다시 시작합니다.
# 이 경로는 이전 컨텍스트에서 이미 사용되었습니다:
initial_best_model_path_in_drive = '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring/weights/best.pt'

print(f"이전 학습의 최적 모델 로딩 중: {initial_best_model_path_in_drive}")
model = YOLO(initial_best_model_path_in_drive) # 초기 최적 모델을 로드합니다.

print("학습을 재개합니다. 이번에는 결과가 Google Drive에 저장될 것입니다...")
model.train(
    data='/content/drive/MyDrive/안전모/dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    cache=True,
    workers=8,
    amp=True,
    device=0,
    # 중요: 미세 조정 결과를 Google Drive에 영구적으로 저장하도록 project 및 name을 구성합니다.
    # 이는 '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring_finetune' 아래에 결과들을 생성합니다.
    project='/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection', # Drive 내 project의 기본 디렉토리
    name='yolo26_safety_monitoring_finetune', # 이 미세 조정 실행을 위한 특정 하위 폴더 이름
    resume=True # 이 옵션은 Drive의 지정된 project/name 경로 내에서 last.pt를 찾습니다.
)

### 학습된 모델로 테스트 데이터셋 예측

In [ ]:
import os
from ultralytics import YOLO

# Google Drive 내 예상되는 모델 가중치 경로
drive_model_path = '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring/weights/best.pt'

print(f"Google Drive 경로 '{drive_model_path}'에 'best.pt' 파일 존재 여부: {os.path.exists(drive_model_path)}")

if os.path.exists(drive_model_path):
    print("\n모델 가중치 파일이 Google Drive에서 확인되었습니다. 이 경로로 예측을 재시도합니다.")

    # 학습된 모델 로드 (Google Drive 경로 사용)
    model = YOLO(drive_model_path)

    # 테스트 이미지 폴더 경로
    test_image_dir = '/content/drive/MyDrive/안전모/dataset/test/images'

    # 예측 수행
    # save=True 로 설정하면 예측 결과 이미지들이 자동으로 저장됩니다.
    # conf와 iou 값은 필요에 따라 조절하세요.
    results = model.predict(source=test_image_dir, save=True, conf=0.25, iou=0.7)

    print("\n테스트 데이터셋에 대한 예측이 완료되었습니다. 결과는 Colab의 runs/detect 폴더 내에 저장될 것입니다.")
    print("참고: 예측 결과는 모델 학습 때와 동일하게 Colab의 임시 디렉토리(예: `/content/runs/detect/predict`)에 저장됩니다.")
else:
    print("\n경고: Google Drive 경로에서도 'best.pt' 파일을 찾을 수 없습니다. 모델이 다른 경로에 저장되었거나, 재학습이 필요할 수 있습니다.")
    print("혹은, 학습 코드를 실행할 때 `project` 인자를 `/content/drive/MyDrive/안전모/runs` 와 같이 Drive 경로로 직접 지정했어야 학습 결과가 Drive에 지속적으로 저장됩니다.")

### 예측 결과 객체(`results`) 분석 및 저장 경로 확인

In [ ]:
import os

# 'results' 객체가 커널 상태에 있으므로 직접 접근 가능합니다.
# sample_results = results[0] # 첫 번째 이미지에 대한 결과

print("--- 샘플 이미지에 대한 감지 결과 확인 ---")

if results:
    # 몇 개의 샘플 이미지 결과만 확인합니다.
    for i, r in enumerate(results[:5]): # 처음 5개 이미지 결과만 확인
        num_detections = len(r.boxes)
        print(f"이미지 {i+1} ({os.path.basename(r.path)}): {num_detections}개의 객체 감지됨")

        # 각 감지된 객체의 클래스 ID와 confidence 출력
        if num_detections > 0:
            for j, box in enumerate(r.boxes):
                cls_id = int(box.cls.cpu().numpy()[0])
                conf = box.conf.cpu().numpy()[0]
                print(f"  - 감지 {j+1}: 클래스 ID {cls_id} (conf: {conf:.2f})")
        else:
            print(f"  - 감지된 객체 없음.")
else:
    print("감지 결과 (results 객체)가 비어있습니다.")

print("\n--- 저장된 예측 이미지 경로 확인 ---")
# ultralytics의 predict 함수는 'save=True'일 때, project/name 인자가 없으면
# 기본적으로 /content/runs/detect/predict 에 결과를 저장합니다.
# 만약 사용자가 수동으로 옮겼다면 해당 경로를 알려주셔야 합니다.

expected_colab_output_dir = '/content/runs/detect/predict'
user_mentioned_drive_path = '/content/drive/MyDrive/안전모/runs_test'

print(f"이전 `model.predict` 명령에 의해 예상되는 Colab 임시 저장 경로: {expected_colab_output_dir}")
print(f"사용자께서 말씀하신 Google Drive 경로: {user_mentioned_drive_path}")

if os.path.exists(expected_colab_output_dir) and os.listdir(expected_colab_output_dir):
    print(f"\n{expected_colab_output_dir} 디렉토리에 결과 이미지가 존재합니다.")
    print(f"일부 파일 목록: {os.listdir(expected_colab_output_dir)[:5]}...")
else:
    print(f"\n경고: {expected_colab_output_dir} 디렉토리가 비어있거나 존재하지 않습니다.")

if os.path.exists(user_mentioned_drive_path) and os.listdir(user_mentioned_drive_path):
    print(f"\n{user_mentioned_drive_path} 디렉토리에도 결과 이미지가 존재합니다.")
    print(f"일부 파일 목록: {os.listdir(user_mentioned_drive_path)[:5]}...")
    print("**사용자님, 혹시 Colab 임시 저장 경로의 내용을 이 Drive 경로로 직접 옮기셨는지요?**")
else:
    print(f"\n경고: {user_mentioned_drive_path} 디렉토리가 비어있거나 존재하지 않습니다.")

print("\n모델은 각 이미지에서 여러 객체를 감지할 수 있지만, 저장된 이미지에서 시각화되는 방식에 따라 표시되는 방식이 달라 보일 수 있습니다.")
print("만약 `results` 객체에서 여러 개의 객체가 감지되었다고 나오는데도 저장된 이미지에서 하나만 보인다면, 이는 주로 시각화 설정의 문제일 가능성이 높습니다.")


### 감지된 모든 객체가 표시된 이미지 시각화 및 Google Drive에 저장

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

# 예측 결과 이미지를 저장할 Drive 경로 설정
output_drive_path = '/content/drive/MyDrive/안전모/runs_test/all_detections'
os.makedirs(output_drive_path, exist_ok=True)

print(f"감지된 모든 객체가 표시된 이미지를 '{output_drive_path}'에 저장합니다.\n")

sample_display_count = 0
for i, r in enumerate(results):
    # r.plot()은 감지된 객체들이 그려진 이미지를 numpy 배열로 반환합니다.
    # 이 배열은 RGB가 아닌 BGR 포맷일 수 있으므로 matplotlib으로 표시하기 위해 변환해야 할 수도 있습니다.
    # ultralytics results 객체의 plot() 메서드는 기본적으로 원본 이미지에 bounding box를 그려 반환합니다.
    im_array = r.plot()  # plot()은 이미지를 BGR numpy 배열로 반환합니다.
    im_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB) # RGB로 변환

    # 파일명은 원본 이미지 파일명 사용
    original_filename = os.path.basename(r.path)
    save_filepath = os.path.join(output_drive_path, original_filename)

    # 저장할 때는 PIL 또는 cv2.imwrite 사용 (cv2가 BGR을 기본으로 하므로 그대로 저장)
    cv2.imwrite(save_filepath, im_array) # BGR 이미지를 그대로 저장

    if sample_display_count < 3: # 처음 3개 이미지만 Colab에 직접 표시
        plt.figure(figsize=(10, 8))
        plt.imshow(im_rgb)
        plt.title(f"Image: {original_filename} (Detections: {len(r.boxes)})")
        plt.axis('off')
        plt.show()
        sample_display_count += 1

    print(f"저장 완료: {save_filepath}")

print(f"\n총 {len(results)}개의 예측 결과 이미지가 '{output_drive_path}'에 저장되었습니다.")
print(f"Google Drive에서 직접 모든 이미지를 확인하실 수 있습니다.")


### 동영상에서 안전모 감지 및 결과 동영상 저장

In [ ]:
# 모델은 이미 이전 단계에서 로드되어 `model` 변수에 저장되어 있습니다.
# model = YOLO(drive_model_path) # 이 줄은 이미 실행되었으므로 다시 실행할 필요 없습니다.

# 동영상 파일 경로를 지정해주세요.
# 예시: '/content/drive/MyDrive/my_test_video.mp4'
video_path = '/content/drive/MyDrive/안전모/sample_video.mp4' # 여기에 사용하실 동영상 파일의 경로를 입력하세요.

if not os.path.exists(video_path):
    print(f"오류: 지정된 동영상 파일 '{video_path}'를 찾을 수 없습니다. 경로를 다시 확인해주세요.")
else:
    print(f"동영상 '{video_path}'에서 안전모 감지를 시작합니다...")

    # 동영상에 대한 예측 수행
    # save=True 로 설정하면 예측 결과 동영상 및 이미지들이 자동으로 저장됩니다.
    # show=True 를 추가하면 Colab에서 직접 결과를 실시간으로 볼 수 있지만, 성능 저하 및 런타임 충돌 가능성이 있습니다.
    video_results = model.predict(source=video_path, save=True, conf=0.25, iou=0.7)

    print("\n동영상 예측이 완료되었습니다.")
    print("결과 동영상은 Colab의 임시 디렉토리(예: `/content/runs/detect/predict`)에 저장될 수 있습니다.")
    print("정확한 저장 경로는 콘솔 출력 메시지에서 `Results saved to` 부분을 확인해주세요.")


이 코드를 실행하시면, 감지된 안전모가 바운딩 박스로 표시된 새로운 동영상 파일이 생성됩니다. 이 파일은 보통 `/content/runs/detect/predict`와 같은 경로에 저장되므로, 필요에 따라 Google Drive로 이동시키시면 됩니다.

### 유튜브 동영상으로 객체 탐지 (yt-dlp 활용)

유튜브 동영상에서 직접 객체 탐지를 수행하려면, 먼저 해당 동영상을 다운로드해야 합니다. `yt-dlp`는 유튜브 동영상을 쉽게 다운로드할 수 있는 강력한 도구입니다. 아래 코드는 `yt-dlp`를 설치하고, 지정된 유튜브 링크에서 동영상을 다운로드하여 객체 탐지에 활용하는 방법을 보여줍니다.

In [ ]:
# yt-dlp 설치
!pip install yt-dlp

In [ ]:
# 다운로드할 유튜브 동영상 링크를 지정하세요.
youtube_url = 'https://youtu.be/EDRtsT_VRD4?si=285BTBp2AECnr86e'  # 여기에 실제 유튜브 링크를 붙여넣으세요.

# 다운로드될 파일명 설정 (예: 'youtube_video.mp4')
downloaded_video_filename = 'youtube_video.mp4'
downloaded_video_path = f'/content/{downloaded_video_filename}'

# yt-dlp를 사용하여 동영상 다운로드
# -f 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best' 옵션은 가능한 최고 화질의 mp4 파일을 다운로드하도록 시도합니다.
!yt-dlp -f 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best' -o "{downloaded_video_path}" "{youtube_url}"

print(f"유튜브 동영상 '{youtube_url}'이 '{downloaded_video_path}'에 다운로드되었습니다.")

# 이제 이 다운로드된 동영상 경로를 사용하여 모델 예측을 수행할 수 있습니다.
# 다음 셀에서 `video_path` 변수를 이 경로로 업데이트하여 실행할 수 있습니다.

In [ ]:
!pip install ultralytics

import os
from ultralytics import YOLO

# Google Drive 내 예상되는 모델 가중치 경로
drive_model_path = '/content/drive/MyDrive/안전모/runs/detect/Helmet_Detection/yolo26_safety_monitoring/weights/best.pt'

# 학습된 모델 로드 (Google Drive 경로 사용)
# 'model' 변수가 정의되지 않았을 수 있으므로 여기서 다시 로드합니다.
model = YOLO(drive_model_path)

# 다운로드된 동영상의 경로를 객체 탐지에 사용할 `video_path` 변수에 할당합니다.
video_path = downloaded_video_path

print(f"객체 탐지를 위해 설정된 동영상 경로: {video_path}")

# 이제 이 `video_path` 변수를 사용하여 다음 셀에서 `model.predict`를 실행할 수 있습니다.
# 다음은 객체 탐지를 수행할 코드입니다.

if not os.path.exists(video_path):
    print(f"오류: 지정된 동영상 파일 '{video_path}'를 찾을 수 없습니다. 경로를 다시 확인해주세요.")
else:
    print(f"동영상 '{video_path}'에서 안전모 감지를 시작합니다...")

    # 동영상에 대한 예측 수행
    # save=True 로 설정하면 예측 결과 동영상 및 이미지들이 자동으로 저장됩니다.
    # show=True 를 추가하면 Colab에서 직접 결과를 실시간으로 볼 수 있지만, 성능 저하 및 런타임 충돌 가능성이 있습니다.
    video_results = model.predict(source=video_path, save=True, conf=0.25, iou=0.7)

    print("\n동영상 예측이 완료되었습니다.")
    print("결과 동영상은 Colab의 임시 디렉토리(예: `/content/runs/detect/predict`)에 저장될 수 있습니다.")
    print("정확한 저장 경로는 콘솔 출력 메시지에서 `Results saved to` 부분을 확인해주세요.")

### 새로운 데이터셋 추가 및 압축 해제

새로운 학습 데이터를 모델에 추가하기 위해 제공된 압축 파일을 지정된 경로에 해제합니다. 이미지 파일과 JSON 라벨 파일을 각각 해당 경로에 위치시킵니다.

In [ ]:
import os

# 압축 해제 대상 경로 설정
extract_base_path = '/content/drive/MyDrive/안전모/안전모 압축해제'
os.makedirs(extract_base_path, exist_ok=True)

# 이미지 압축 파일 경로
image_zip_path = '/content/drive/MyDrive/안전모/알집/202105_안전모 (1).zip'
# JSON 압축 파일 경로
json_zip_path = '/content/drive/MyDrive/안전모/알집/202105_안전모_json.zip'

print(f"'{image_zip_path}' 압축 해제 중...")
!unzip -q "{image_zip_path}" -d "{extract_base_path}/images"

print(f"'{json_zip_path}' 압축 해제 중...")
!unzip -q "{json_zip_path}" -d "{extract_base_path}/json_labels"

print(f"모든 새로운 데이터가 '{extract_base_path}' 폴더에 성공적으로 압축 해제되었습니다.")

### 새로 추가된 데이터셋의 이미지-JSON 라벨 매칭 확인

새로 압축 해제된 이미지와 JSON 라벨 파일들이 각각 올바르게 존재하는지 확인합니다. 이 단계는 데이터의 무결성을 검증하고, 나중에 YOLO 형식으로 변환할 라벨 데이터를 준비하는 데 중요합니다.

In [ ]:
import os

# 압축 해제된 이미지 및 JSON 라벨 폴더 경로 설정
# extract_base_path는 이전 셀에서 정의되었습니다.
image_dir = os.path.join(extract_base_path, 'images')
json_label_dir = os.path.join(extract_base_path, 'json_labels')

# 이미지 파일 목록 가져오기
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
image_names = {os.path.splitext(f)[0] for f in image_files}

# JSON 라벨 파일 목록 가져오기
json_files = [f for f in os.listdir(json_label_dir) if f.lower().endswith('.json')]
json_names = {os.path.splitext(f)[0] for f in json_files}

print(f"새로 압축 해제된 이미지 파일 수: {len(image_files)}")
print(f"새로 압축 해제된 JSON 라벨 파일 수: {len(json_files)}")

# 이미지와 JSON 라벨 매칭 확인
matched_count = 0
images_without_json = []
json_without_images = []

for img_name in image_names:
    if img_name in json_names:
        matched_count += 1
    else:
        images_without_json.append(img_name)

for json_name in json_names:
    if json_name not in image_names:
        json_without_images.append(json_name)

print(f"\n매칭되는 이미지-JSON 라벨 쌍 수: {matched_count}")

if images_without_json:
    print(f"\nJSON 라벨이 없는 이미지 ({len(images_without_json)}개):")
    for img in images_without_json[:10]: # 처음 10개만 출력
        print(f"  - {img}.jpg (또는 .png)")
    if len(images_without_json) > 10:
        print(f"  ...외 {len(images_without_json) - 10}개")
else:
    print("\nJSON 라벨이 없는 이미지는 없습니다.")

if json_without_images:
    print(f"\n이미지 파일이 없는 JSON 라벨 ({len(json_without_images)}개):")
    for json_file in json_without_images[:10]: # 처음 10개만 출력
        print(f"  - {json_file}.json")
    if len(json_without_images) > 10:
        print(f"  ...외 {len(json_without_images) - 10}개")
else:
    print("\n이미지 파일이 없는 JSON 라벨은 없습니다.")

if matched_count == len(image_files) and matched_count == len(json_files):
    print("\n모든 이미지 파일과 JSON 라벨 파일이 완벽하게 매칭됩니다. 데이터셋 준비가 순조롭습니다.")
else:
    print("\n일부 이미지 또는 JSON 라벨 파일이 매칭되지 않습니다. 위 목록을 확인하여 불일치하는 파일을 처리해야 할 수 있습니다.")

### 새로운 JSON 라벨을 YOLO 형식으로 변환

이제 압축 해제된 JSON 라벨 파일들을 YOLO 모델 학습에 필요한 `.txt` 형식으로 변환해야 합니다. `convert_to_yolo` 함수를 사용하여 이 작업을 수행하고, 변환된 라벨 파일은 새로운 폴더에 저장하겠습니다. 이 과정에서 `class_mapping`이 올바르게 설정되어 있는지 확인해주세요.

In [ ]:
import json
import os

# 클래스 맵핑 (안전모: 1, 머리: 0 등 프로젝트 설정에 맞춰주세요)
# '안전모'와 '머리'로 클래스 매핑 업데이트
class_mapping = {"안전모": 1, "머리": 0}

def convert_to_yolo(json_data, output_path):
    # Ensure img_w and img_h are correctly extracted and are not zero
    if not json_data.get('images') or not json_data['images'][0].get('width') or not json_data['images'][0].get('height'):
        print(f"Warning: Image dimensions missing in JSON data for {output_path}. Skipping file.")
        return

    img_w = float(json_data['images'][0]['width'])
    img_h = float(json_data['images'][0]['height'])

    if img_w <= 0 or img_h <= 0:
        print(f"Warning: Invalid image dimensions (w={img_w}, h={img_h}) for {output_path}. Skipping file.")
        return

    yolo_lines = []
    annotations_processed = 0
    annotations_skipped_invalid_class = 0
    annotations_skipped_invalid_polygon = 0
    annotations_skipped_invalid_bbox = 0

    for ann in json_data['annotations']:
        cls_name = ann.get('class') # Use .get to safely access
        if cls_name not in class_mapping:
            annotations_skipped_invalid_class += 1
            # print(f"Debug: Skipped annotation in {os.path.basename(output_path)} due to unknown class: '{cls_name}'")
            continue

        cls_id = class_mapping[cls_name]

        polygon = ann.get('polygon')
        if not polygon or len(polygon) < 4: # A polygon needs at least 2 points (4 coordinates) to form a box
            annotations_skipped_invalid_polygon += 1
            print(f"Debug: Skipped annotation in {os.path.basename(output_path)} due to invalid polygon data: {polygon}") # Debug print for invalid polygon
            continue

        x_coords = [float(p) for p in polygon[0::2]] # Ensure coordinates are floats
        y_coords = [float(p) for p in polygon[1::2]]

        # Calculate Bounding Box
        # Clip to 0 and img_w-1/img_h-1 to ensure coordinates are within image boundaries
        xmin_raw = min(x_coords)
        ymin_raw = min(y_coords)
        xmax_raw = max(x_coords)
        ymax_raw = max(y_coords)

        xmin = max(0.0, xmin_raw)
        ymin = max(0.0, ymin_raw)
        xmax = min(img_w - 1, xmax_raw)
        ymax = min(img_h - 1, ymax_raw)

        # Check for valid bounding box after clipping
        if xmax <= xmin or ymax <= ymin:
            annotations_skipped_invalid_bbox += 1
            # print(f"Debug: Skipped annotation in {output_path} due to degenerate bbox after clipping: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}, img_w={img_w}, img_h={img_h}")
            continue

        # YOLO 포맷 변환 (정규화 및 중심점 계산)
        x_center = ((xmin + xmax) / 2.0) / img_w
        y_center = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        # Ensure normalized values are strictly within [0, 1] range
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        w = max(0.0, min(1.0, w))
        h = max(0.0, min(1.0, h))

        yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        annotations_processed += 1

    # Only write the file if there are valid annotations, otherwise create an empty file
    with open(output_path, 'w') as f:
        if yolo_lines:
            f.write("\n".join(yolo_lines))

    if annotations_skipped_invalid_class > 0 or annotations_skipped_invalid_polygon > 0 or annotations_skipped_invalid_bbox > 0:
        print(f"Summary for {os.path.basename(output_path)}: Processed {annotations_processed} annotations. Skipped: {annotations_skipped_invalid_class} (unknown class), {annotations_skipped_invalid_polygon} (invalid polygon), {annotations_skipped_invalid_bbox} (invalid bbox after clipping).")

# YOLO 라벨을 저장할 새로운 디렉토리 설정
new_yolo_label_dir = os.path.join(extract_base_path, 'yolo_labels')
os.makedirs(new_yolo_label_dir, exist_ok=True)

print(f"새로운 JSON 라벨 파일을 YOLO 형식으로 변환하여 '{new_yolo_label_dir}'에 저장합니다...")

# 새로 압축 해제된 JSON 라벨 파일들 처리
newly_extracted_json_files = [f for f in os.listdir(json_label_dir) if f.endswith('.json')]

processed_count = 0
for j_file in newly_extracted_json_files:
    json_filepath = os.path.join(json_label_dir, j_file)
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 이미지 파일명에서 라벨 파일명 유추 (확장자 변경)
    txt_filename = os.path.splitext(j_file)[0] + '.txt'
    save_path = os.path.join(new_yolo_label_dir, txt_filename)

    convert_to_yolo(data, save_path)
    processed_count += 1

print(f"총 {processed_count}개의 JSON 라벨 파일이 YOLO 형식으로 변환 완료되었습니다.")
print(f"변환된 라벨 파일은 '{new_yolo_label_dir}'에서 확인하실 수 있습니다.")

# 변환된 라벨 파일 생성 확인
yolo_label_files = [f for f in os.listdir(new_yolo_label_dir) if f.endswith('.txt')]
print(f"생성된 YOLO 라벨 파일 수: {len(yolo_label_files)} 개")

### 새로 변환된 YOLO 라벨 파일 샘플 확인

In [ ]:
import random
import os

# 새로 생성된 YOLO 라벨 파일 목록
yolo_label_files = [f for f in os.listdir(new_yolo_label_dir) if f.endswith('.txt')]

print(f"총 생성된 YOLO 라벨 파일 수: {len(yolo_label_files)} 개")

# 비어있지 않은 라벨 파일 수 확인
non_empty_yolo_label_files = 0
for l_file in yolo_label_files:
    with open(os.path.join(new_yolo_label_dir, l_file), 'r', encoding='utf-8') as f:
        content = f.read()
        if content.strip(): # 내용이 비어있지 않은 경우
            non_empty_yolo_label_files += 1

print(f"성공적으로 라벨이 변환된 (비어있지 않은) 파일 수: {non_empty_yolo_label_files} 개")

if non_empty_yolo_label_files == 0:
    print("경고: 생성된 YOLO 라벨 파일 중 유효한 내용이 있는 파일이 없습니다. 라벨링 데이터 또는 변환 로직을 다시 확인해주세요.")

# 랜덤으로 5개 파일 내용 출력
print("\n--- 랜덤 샘플 YOLO 라벨 파일 내용 (5개) ---")
if yolo_label_files:
    sample_files = random.sample(yolo_label_files, min(5, len(yolo_label_files)))
    for s_file in sample_files:
        print(f"\n파일: {s_file}")
        with open(os.path.join(new_yolo_label_dir, s_file), 'r', encoding='utf-8') as f:
            print(f.read().strip())
else:
    print("생성된 YOLO 라벨 파일이 없습니다.")

### 새로운 데이터셋을 훈련 데이터셋으로 통합

새로 압축 해제하고 YOLO 형식으로 변환한 이미지 파일과 라벨 파일들을 기존의 훈련 데이터셋(`train/images`, `train/labels`) 폴더로 이동시켜 모델 추가 학습에 활용할 수 있도록 준비합니다.

In [ ]:
import os
import shutil

# 기존에 정의된 extract_base_path를 사용합니다.
# extract_base_path = '/content/drive/MyDrive/안전모/안전모 압축해제'

source_images_dir = os.path.join(extract_base_path, 'images')
source_yolo_labels_dir = os.path.join(extract_base_path, 'yolo_labels')

# 목표 훈련 데이터셋 경로 설정 (data.yaml에 명시된 경로와 일치해야 합니다)
target_train_images_dir = '/content/drive/MyDrive/안전모/dataset/train/images'
target_train_labels_dir = '/content/drive/MyDrive/안전모/dataset/train/labels'

# 목표 디렉토리가 없으면 생성
os.makedirs(target_train_images_dir, exist_ok=True)
os.makedirs(target_train_labels_dir, exist_ok=True)

print(f"새로운 이미지를 '{target_train_images_dir}'로 이동 중...")
image_files_to_move = os.listdir(source_images_dir)
for f in image_files_to_move:
    shutil.move(os.path.join(source_images_dir, f), os.path.join(target_train_images_dir, f))
print(f"{len(image_files_to_move)}개의 이미지 파일 이동 완료.")

print(f"새로운 YOLO 라벨을 '{target_train_labels_dir}'로 이동 중...")
label_files_to_move = os.listdir(source_yolo_labels_dir)
for f in label_files_to_move:
    shutil.move(os.path.join(source_yolo_labels_dir, f), os.path.join(target_train_labels_dir, f))
print(f"{len(label_files_to_move)}개의 YOLO 라벨 파일 이동 완료.")

print("\n데이터셋 통합이 완료되었습니다. 이제 모델 추가 학습을 시작할 준비가 되었습니다.")